In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def collect_laacib_links(base_url, total_pages):
    all_links = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    for page_num in range(1, total_pages + 1):
        # Construct the paginated URL
        url = f"{base_url}page/{page_num}/"
        print(f"Fetching: {url}")
        
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Find all <a> tags with rel="bookmark"
                # This targets the specific post links you identified
                post_links = soup.find_all('a', rel='bookmark')
                
                page_links_count = 0
                for link in post_links:
                    href = link.get('href')
                    title = link.get('title') # Also grabbing the title for your NLP context
                    
                    if href and href not in [l['link'] for l in all_links]:
                        all_links.append({
                            'title': title,
                            'link': href,
                            'category': 'ciyaaro' # Tagging it for your dataset
                        })
                        page_links_count += 1
                
                print(f"Found {page_links_count} new links on page {page_num}.")
            else:
                print(f"Failed to load page {page_num}. Status: {response.status_code}")
                break # Stop if we hit a page that doesn't exist
                
        except Exception as e:
            print(f"Error on page {page_num}: {e}")
            continue
            
        # Be polite to the server to avoid being blocked
        time.sleep(2)

    return all_links

# --- CONFIGURATION ---
# The base category URL (without the 'page/X/')
target_base_url = "https://www.laacibnet.net/category/wararka-premier-league/"
total_pages_to_scrape = 5  # Change this to however many pages you need

if __name__ == "__main__":
    links_data = collect_laacib_links(target_base_url, total_pages_to_scrape)
    
    # Save to Excel
    if links_data:
        df = pd.DataFrame(links_data)
        output_filename = "laacibnet_premier_league_links.xlsx"
        df.to_excel(output_filename, index=False)
        print(f"\nSuccess! Saved {len(links_data)} unique links to {output_filename}")
    else:
        print("No links collected.")

Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/1/
Found 17 new links on page 1.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/2/
Found 12 new links on page 2.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/3/
Found 12 new links on page 3.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/4/
Found 12 new links on page 4.
Fetching: https://www.laacibnet.net/category/wararka-premier-league/page/5/
Found 12 new links on page 5.

Success! Saved 65 unique links to laacibnet_premier_league_links.xlsx


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

def collect_links_with_stagnation_check(categories_dict, output_file, max_stagnation=5):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Load existing data for Checkpoint
    if os.path.exists(output_file):
        df_master = pd.read_excel(output_file)
        all_links = df_master.to_dict('records')
        existing_urls = set(df_master['link'].tolist())
        print(f"Resuming: {len(existing_urls)} links already in database.")
    else:
        all_links = []
        existing_urls = set()

    for category_name, base_url in categories_dict.items():
        print(f"\n--- Processing Category: {category_name} ---")
        
        page = 1
        stagnation_counter = 0  # Tracks consecutive pages with 0 new links
        
        while True:
            url = f"{base_url}page/{page}/"
            print(f"Fetching Page {page}: {url}")
            
            try:
                response = requests.get(url, headers=headers, timeout=15)
                
                # 1. Stop if the server returns an error code (404, 500, etc.)
                if response.status_code != 200:
                    print(f"Reached end or error (Status: {response.status_code}).")
                    break
                
                soup = BeautifulSoup(response.content, 'html.parser')
                post_links = soup.find_all('a', rel='bookmark')
                
                # 2. Stop if the page is physically empty of links
                if not post_links:
                    print("Page is empty. Moving to next category.")
                    break

                new_on_page = 0
                for link in post_links:
                    href = link.get('href')
                    title = link.get('title')
                    
                    if href and href not in existing_urls:
                        all_links.append({
                            'title': title,
                            'link': href,
                            'category': category_name,
                            'source': 'laacibnet'
                        })
                        existing_urls.add(href)
                        new_on_page += 1
                
                # 3. Stagnation Logic
                if new_on_page == 0:
                    stagnation_counter += 1
                    print(f"Stagnation warning: {stagnation_counter}/{max_stagnation} pages with no new links.")
                else:
                    stagnation_counter = 0  # Reset counter if we find even one new link
                
                if stagnation_counter >= max_stagnation:
                    print(f"Stagnation limit reached for {category_name}. Skipping to next...")
                    break

                print(f"Added {new_on_page} new links from page {page}.")
                
                # Checkpoint Save
                pd.DataFrame(all_links).to_excel(output_file, index=False)
                
                page += 1
                time.sleep(1.5)
                
            except Exception as e:
                print(f"Error on {url}: {e}")
                break

    print(f"\nCompleted! Total unique links: {len(all_links)}")

# --- CONFIGURATION ---
categories = {
    "ciyaaraha_maanta": "https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/",
    "premier_league": "https://www.laacibnet.net/category/wararka-premier-league/",
    "la_liga": "https://www.laacibnet.net/category/wararka-la-liga/"
}

output_excel = "laacibnet_curated_dataset.xlsx"

if __name__ == "__main__":
    collect_links_with_stagnation_check(categories, output_excel)

Resuming: 626 links already in database.

--- Processing Category: ciyaaraha_maanta ---
Fetching Page 1: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/1/
Added 17 new links from page 1.
Fetching Page 2: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/2/
Added 12 new links from page 2.
Fetching Page 3: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/3/
Added 12 new links from page 3.
Fetching Page 4: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/4/
Added 5 new links from page 4.
Fetching Page 5: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/5/
Stagnation warning: 1/5 pages with no new links.
Added 0 new links from page 5.
Fetching Page 6: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/6/
Stagnation warning: 2/5 pages with no new links.
Added 0 new links from page 6.
Fetching Page 7: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/7/
Stagnation warning: 3/5 pages 

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

def collect_all_links(categories_dict, output_file):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    # Load existing data for Checkpoint to avoid starting over
    if os.path.exists(output_file):
        try:
            df_master = pd.read_excel(output_file)
            all_links = df_master.to_dict('records')
            existing_urls = set(df_master['link'].tolist())
            print(f"Resuming: {len(existing_urls)} links already in database.")
        except Exception as e:
            print(f"Error loading existing file: {e}. Starting fresh.")
            all_links = []
            existing_urls = set()
    else:
        all_links = []
        existing_urls = set()

    for category_name, info in categories_dict.items():
        base_url = info['url']
        max_pages = info['pages']
        
        print(f"\n--- Processing Category: {category_name} (Limit: {max_pages} pages) ---")
        
        # Hard limit using the counts you provided
        for page in range(1, max_pages + 1):
            url = f"{base_url}page/{page}/"
            print(f"Fetching Page {page}/{max_pages}: {url}")
            
            try:
                response = requests.get(url, headers=headers, timeout=15)
                
                # Stop if the server returns a 404 or other error
                if response.status_code != 200:
                    print(f"Server returned status {response.status_code}. Moving to next category.")
                    break
                
                soup = BeautifulSoup(response.content, 'html.parser')
                
                # Standard WordPress bookmark link selector
                post_links = soup.find_all('a', rel='bookmark')
                
                if not post_links:
                    print(f"No links found on page {page}. Moving to next category.")
                    break

                new_on_page = 0
                dup_on_page = 0

                for link in post_links:
                    href = link.get('href')
                    # Get title attribute, or fallback to the link text
                    title = link.get('title') or link.get_text(strip=True)
                    
                    if href:
                        if href not in existing_urls:
                            all_links.append({
                                'title': title,
                                'link': href,
                                'category': category_name,
                                'source': 'laacibnet'
                            })
                            existing_urls.add(href)
                            new_on_page += 1
                        else:
                            dup_on_page += 1
                
                print(f"Page {page}: Added {new_on_page} new, Skipped {dup_on_page} duplicates.")
                
                # Checkpoint: Save to Excel after every successful page
                pd.DataFrame(all_links).to_excel(output_file, index=False)
                
                # Polite delay to avoid server blocks
                time.sleep(1.2)
                
            except Exception as e:
                print(f"Error on {url}: {e}")
                # We break the page loop for this category on error but continue to the next category
                break

    print(f"\n✅ Scraping Complete! Final unique link count: {len(all_links)}")

# --- CONFIGURATION ---
# Based on your requirements: 54, 48, and 9 pages respectively.
categories = {
    "ciyaaraha_maanta": {
        "url": "https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/", 
        "pages": 54
    },
    "premier_league": {
        "url": "https://www.laacibnet.net/category/wararka-premier-league/", 
        "pages": 48
    },
    "la_liga": {
        "url": "https://www.laacibnet.net/category/wararka-la-liga/", 
        "pages": 9
    }
}

output_excel = "laacibnet_full_category_links.xlsx"

if __name__ == "__main__":
    collect_all_links(categories, output_excel)


--- Processing Category: ciyaaraha_maanta (Limit: 54 pages) ---
Fetching Page 1/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/1/
Page 1: Added 17 new, Skipped 25 duplicates.
Fetching Page 2/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/2/
Page 2: Added 12 new, Skipped 30 duplicates.
Fetching Page 3/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/3/
Page 3: Added 12 new, Skipped 30 duplicates.
Fetching Page 4/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/4/
Page 4: Added 12 new, Skipped 30 duplicates.
Fetching Page 5/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/5/
Page 5: Added 12 new, Skipped 30 duplicates.
Fetching Page 6/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/6/
Page 6: Added 12 new, Skipped 30 duplicates.
Fetching Page 7/54: https://www.laacibnet.net/category/wararka-ciyaaraha-maanta/page/7/
Page 7: Added 12 new, Skipped 30 duplicates.
Fetc

In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os

def scrape_laacibnet_article(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # 1. Get the Headline (h1 is standard for titles)
            headline = soup.find('h1').get_text(strip=True) if soup.find('h1') else "No Headline Found"
            
            # 2. Get the Body
            # Laacibnet uses 'entry-content' for the main article text
            body_content = ""
            article_div = soup.find('div', class_='entry-content') or \
                          soup.find('div', class_='post-content')
            
            if article_div:
                # Filter out script or style tags if they exist inside the div
                for element in article_div(['script', 'style', 'aside']):
                    element.decompose()
                
                paragraphs = article_div.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
            else:
                # Fallback to all p tags if the specific container is missing
                paragraphs = soup.find_all('p')
                body_content = "\n".join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])

            return headline, body_content
        else:
            return None, f"Error: Status {response.status_code}"
    except Exception as e:
        return None, f"Request failed: {str(e)}"

# 1. Load the links we collected
input_file = "laacibnet_full_category_links.xlsx"
df_links = pd.read_excel(input_file)

# 2. Prepare for results
output_file = "laacibnet_scraped_articles.xlsx"
scraped_data = []

# If you need to resume a partial scrape, you can load existing data here
if os.path.exists(output_file):
    df_existing = pd.read_excel(output_file)
    scraped_urls = set(df_existing['url'].tolist())
    scraped_data = df_existing.to_dict('records')
    print(f"Resuming: {len(scraped_urls)} articles already scraped.")
else:
    scraped_urls = set()

print(f"Starting to scrape {len(df_links)} articles...")

# 3. Main Scraping Loop
for index, row in df_links.iterrows():
    url = row['link']
    
    if url in scraped_urls:
        continue
        
    print(f"[{index + 1}/{len(df_links)}] Scraping: {url}")
    
    headline, body = scrape_laacibnet_article(url)
    
    if headline and body:
        scraped_data.append({
            'url': url,
            'headline': headline,
            'body': body,
            'category': row['category'],
            'source': row['source']
        })
        scraped_urls.add(url)
    
    # Save every 10 articles as a checkpoint
    if len(scraped_data) % 10 == 0:
        pd.DataFrame(scraped_data).to_excel(output_file, index=False)
    
    # Respectful delay
    time.sleep(1.5)

# 4. Final Export
pd.DataFrame(scraped_data).to_excel(output_file, index=False)
print(f"\n✅ Finished! Scraped {len(scraped_data)} articles to {output_file}")

Starting to scrape 647 articles...
[1/647] Scraping: https://www.laacibnet.net/fa-ga-ingiriiska-oo-ganaax-culus-dul-dhigay-mid-kamid-ah-weeraryahannada-chelsea/
[2/647] Scraping: https://www.laacibnet.net/neymar-oo-shaaciyay-labada-dal-ee-uu-doonayo-inay-isugu-yimaadaan-final-ka-koobka-adduunka/
[3/647] Scraping: https://www.laacibnet.net/thierry-henry-oo-kashifay-sirta-guulaha-arsenal-ee-xilli-ciyaareed-kaan-ka-hor-kulanka-atletico-madrid/
[4/647] Scraping: https://www.laacibnet.net/psg-mise-bayern-wayne-rooney-oo-shaaciyay-kooxda-uu-aaminsan-yahay-inay-u-soo-gudbeyso-final-ka-tartanka-champions-league-ga/
[5/647] Scraping: https://www.laacibnet.net/odegaard-oo-shaaca-ka-qaaday-xiddiga-ka-dhiga-arsenal-mid-halis-badan-oo-ay-kooxaha-ka-argagaxaan/
[6/647] Scraping: https://www.laacibnet.net/obi-mikel-oo-shaaciyay-xiddig-chelsea-ah-oo-ku-faaiday-ceydhintii-liam-rosenior/
[7/647] Scraping: https://www.laacibnet.net/warbixin-qorshaha-liverpool-ee-suuqa-xagaaga/
[8/647] Scraping: https://w